# Schema Design, Normalization and Data Warehouse Modeling

Notebook for **Chapter 11** of the *sql* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/sql/ch11.html

Tip: change a number, predict the output, then run the cell again.

## 11.1 ER Diagrams, Keys and Naming

## 11.2 Normalization: 1NF, 2NF, 3NF and BCNF

## 11.3 Denormalization and OLTP vs OLAP

## 11.4 Star and Snowflake Schemas

## 11.5 Slowly Changing Dimensions (SCD)

## 11.6 Modelling Extras: Functional-Dependency Closure, BCNF by Algorithm, and Patterns You Will Meet

In [ ]:
from itertools import combinations
attrs = {"student", "course", "grade", "teacher", "office"}
fds = [({"student", "course"}, {"grade"}), ({"course"}, {"teacher"}), ({"teacher"}, {"office"})]

def closure(x):
    """everything that x determines, by repeatedly applying the rules"""
    x = set(x)
    changed = True
    while changed:
        changed = False
        for left, right in fds:
            if left <= x and not right <= x:
                x |= right
                changed = True
    return x

print("closure of {course}          :", sorted(closure({"course"})))
print("closure of {student,course}  :", sorted(closure({"student", "course"})))
keys = [set(c) for n in range(1, 4) for c in combinations(sorted(attrs), n) if closure(c) == attrs]
minimal = [k for k in keys if not any(o < k for o in keys)]
print("candidate keys:", [sorted(k) for k in minimal])
for left, right in fds:
    ok = closure(left) == attrs
    print("%-18s -> %-8s left side is a superkey: %s" % (",".join(sorted(left)), ",".join(sorted(right)), ok))

closure of {course}          : ['course', 'office', 'teacher']
closure of {student,course}  : ['course', 'grade', 'office', 'student', 'teacher']
candidate keys: [['course', 'student']]
course,student     -> grade    left side is a superkey: True
course             -> teacher  left side is a superkey: False
teacher            -> office   left side is a superkey: False


In [ ]:
import sqlite3
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE enrol (student TEXT, course TEXT, grade TEXT, teacher TEXT, office TEXT);
INSERT INTO enrol VALUES
 ('Asha','SQL','A','Rao','B12'), ('Ravi','SQL','B','Rao','B12'),
 ('Asha','Stats','B','Iyer','C03'), ('Meena','Stats','A','Iyer','C03'),
 ('Ravi','ML','A','Iyer','C03');
-- BCNF decomposition from the closure analysis
CREATE TABLE teacher_office (teacher TEXT PRIMARY KEY, office TEXT);
CREATE TABLE course_teacher (course TEXT PRIMARY KEY, teacher TEXT REFERENCES teacher_office);
CREATE TABLE grade_of (student TEXT, course TEXT REFERENCES course_teacher, grade TEXT, PRIMARY KEY (student, course));
INSERT INTO teacher_office SELECT DISTINCT teacher, office FROM enrol;
INSERT INTO course_teacher SELECT DISTINCT course, teacher FROM enrol;
INSERT INTO grade_of SELECT student, course, grade FROM enrol;
""")
rebuilt = db.execute("""SELECT g.student, g.course, g.grade, c.teacher, t.office
  FROM grade_of g JOIN course_teacher c ON c.course = g.course JOIN teacher_office t ON t.teacher = c.teacher
  ORDER BY 1, 2""").fetchall()
original = db.execute("SELECT * FROM enrol ORDER BY 1, 2").fetchall()
print("rows original:", len(original), "| rows rebuilt:", len(rebuilt), "| identical:", rebuilt == original)
db.execute("UPDATE teacher_office SET office = 'D07' WHERE teacher = 'Iyer'")   # one place to change
print(db.execute("SELECT DISTINCT course, office FROM course_teacher c JOIN teacher_office t USING (teacher) ORDER BY 1").fetchall())

rows original: 5 | rows rebuilt: 5 | identical: True
[('ML', 'D07'), ('SQL', 'B12'), ('Stats', 'D07')]


In [ ]:
import sqlite3
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE users (id INTEGER PRIMARY KEY, email TEXT NOT NULL, deleted_at TEXT);
CREATE UNIQUE INDEX users_email_live ON users(email) WHERE deleted_at IS NULL;
INSERT INTO users (email) VALUES ('a@x.com');
UPDATE users SET deleted_at = '2024-05-01' WHERE email = 'a@x.com';
INSERT INTO users (email) VALUES ('a@x.com');     -- allowed: the old row is soft-deleted
""")
print(db.execute("SELECT id, email, deleted_at FROM users ORDER BY id").fetchall())
try:
    db.execute("INSERT INTO users (email) VALUES ('a@x.com')")   # a second live row with the same email
except sqlite3.IntegrityError as e:
    print("blocked:", e)
print("live users:", db.execute("SELECT count(*) FROM users WHERE deleted_at IS NULL").fetchone()[0])

[(1, 'a@x.com', '2024-05-01'), (2, 'a@x.com', None)]
blocked: UNIQUE constraint failed: users.email
live users: 1


In [ ]:
import sqlite3
db = sqlite3.connect(":memory:")
db.execute("PRAGMA foreign_keys = ON")
db.executescript("""
CREATE TABLE customer (tenant_id INTEGER, customer_id INTEGER, name TEXT, PRIMARY KEY (tenant_id, customer_id));
CREATE TABLE orders (
  tenant_id INTEGER, order_id INTEGER, customer_id INTEGER,
  PRIMARY KEY (tenant_id, order_id),
  FOREIGN KEY (tenant_id, customer_id) REFERENCES customer (tenant_id, customer_id));
INSERT INTO customer VALUES (1, 10, 'Asha'), (2, 10, 'Zed');
INSERT INTO orders VALUES (1, 100, 10);     -- tenant 1 order for tenant 1 customer: fine
""")
try:
    db.execute("INSERT INTO orders VALUES (1, 101, 99)")      # tenant 1 pointing at a customer tenant 1 does not have
except sqlite3.IntegrityError as e:
    print("blocked:", e)
print(db.execute("SELECT * FROM orders").fetchall())

blocked: FOREIGN KEY constraint failed
[(1, 100, 10)]


In [ ]:
import sqlite3
db = sqlite3.connect(":memory:")
db.execute("PRAGMA foreign_keys = ON")
db.executescript("""
CREATE TABLE post (post_id INTEGER PRIMARY KEY);
CREATE TABLE photo (photo_id INTEGER PRIMARY KEY);
CREATE TABLE comment (
  comment_id INTEGER PRIMARY KEY,
  post_id  INTEGER REFERENCES post,
  photo_id INTEGER REFERENCES photo,
  body TEXT,
  CHECK ((post_id IS NOT NULL) + (photo_id IS NOT NULL) = 1)   -- exactly one parent
);
INSERT INTO post VALUES (1); INSERT INTO photo VALUES (1);
INSERT INTO comment (post_id, body) VALUES (1, 'nice post');
""")
for sql in ["INSERT INTO comment (post_id, photo_id, body) VALUES (1, 1, 'two parents')",
            "INSERT INTO comment (post_id, body) VALUES (77, 'ghost post')",
            "INSERT INTO comment (body) VALUES ('orphan')"]:
    try:
        db.execute(sql); print("accepted:", sql[:44])
    except sqlite3.IntegrityError as e:
        print("blocked :", e)

blocked : CHECK constraint failed: (post_id IS NOT NULL) + (photo_id IS NOT NULL) = 1
blocked : FOREIGN KEY constraint failed
blocked : CHECK constraint failed: (post_id IS NOT NULL) + (photo_id IS NOT NULL) = 1


## Chapter recap

## Practice set

Problem 11.P.1 (easy) Where does the foreign key go: one `author` writes many `books`? **Answer:** On `books` (`books.author_id`), the many side.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/sql/ch11.html